In [9]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
from pathlib import Path
plt.rcParams["font.size"] = 16

un panel avec des courbes <x>(t) pour different l dans le cas périodique pour le cas mu,theta=160,20

un panel avec des courbes <x>(t) pour different l dans le cas aléatoire pour le cas mu,theta=160,20

un panel avec des courbes <x>(t) pour different bpmin dans le cas aléatoire pour le cas mu,theta=160,20

un panel avec des courbes v_init vs mu pour différent l dans le cas périodique pour le cas theta=20

un panel avec des courbes v_init vs mu pour différent l dans le cas aléatoire pour le cas theta=20

un panel avec des courbes v_init vs mu pour différent bpmin dans le cas aléatoire pour le cas theta=20

In [ ]:
root  = Path("/home/nicolas/Documents/Workspace/nucleo/outputs/NUCLEO__PSMN__2026-06-19")
paths = [str(p) for p in root.glob("nucleo__*")]

mu    = 160
theta = 20
s_selected = 150
ls = np.array([5, 20, 50, 100, 150])
ylims = [-500, 7_000]

ARRAY_COLS = ["times", "results_mean"]

df_data = (
    pl.scan_parquet(paths)
    .filter(
        (pl.col("mu")    == mu)    &
        (pl.col("theta") == theta) &
        (pl.col("s")     == s_selected) &
        (pl.col("l").is_in(ls))
    )
    .select(cs.numeric() | cs.boolean() | cs.string() | cs.by_name(*ARRAY_COLS))
    .collect()
    .sort(["land", "bpmin", "l"])
)

print(df_data)
print(df_data.columns)

In [ ]:
df_periodic = df_data.filter(pl.col("land") == "periodic")

groups = (
    df_periodic
    .group_by("l")
    .agg([
        pl.col("times").first(),
        pl.col("results_mean").first()
    ])
    .sort("l")
)

plt.figure(figsize=(8, 6), dpi=1200)

for row in groups.iter_rows(named=True):
    l = row["l"]
    x = row["times"]
    y = row["results_mean"]

    plt.plot(x, y, label=rf"$l={l}$")

plt.grid(True, alpha=0.3)
plt.title("Periodic")
plt.xlabel(r"Time t ($1/k_0$)")
plt.ylabel("Position")
plt.ylim(ylims)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
df_random = df_data.filter(pl.col("land") == "random")

groups = (
    df_random
    .group_by("l")
    .agg([
        pl.col("times").first(),
        pl.col("results_mean").first()
    ])
    .sort("l")
)

plt.figure(figsize=(8, 6), dpi=1200)

for row in groups.iter_rows(named=True):
    l = row["l"]
    x = row["times"]
    y = row["results_mean"]

    plt.plot(x, y, label=rf"$l={l}$")

plt.grid(True, alpha=0.3)
plt.title("Random")
plt.xlabel(r"Time t ($1/k_0$)")
plt.ylabel("Position")
plt.ylim(ylims)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
df_homogen = df_data.filter(pl.col("land") == "periodic")

groups = (
    df_homogen
    .group_by("l")
    .agg([
        pl.col("times").first(),
    ])
    .sort("l")
)

plt.figure(figsize=(8, 6), dpi=1200)

for row in groups.iter_rows(named=True):
    l = row["l"]
    x = np.array(row["times"])
    y = mu * (l / (s_selected + l))


    plt.plot(x, x * y, label=rf"$l={l}$")

plt.grid(True, alpha=0.3)
plt.title("Homogen")
plt.xlabel(r"Time t ($1/k_0$)")
plt.ylabel("Position")
plt.ylim(ylims)
plt.tight_layout()
plt.legend()
plt.show()

# .